In [1]:
# 1. Google Drive'ı Bağla
from google.colab import drive
drive.mount('/content/drive')

# 2. Ultralytics (YOLO) Kütüphanesini Kur
%pip install ultralytics

# 3. GPU Kontrolü (Ekranda 'True' ve 'Tesla T4' görmelisin)
import torch
print(f"GPU Aktif mi?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Model: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU SEÇİLMEMİŞ! Lütfen Çalışma Zamanı ayarlarından T4 GPU'yu seç.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 86.7 MB/s eta 0:00:00
GPU Aktif mi?: True
Model: Tesla T4


In [2]:
# ==========================================
# TEK HÜCREDE EĞİTİM VE DÖNÜŞÜM
# ==========================================
import os
from ultralytics import YOLO

# 1. AYARLAR
# ------------------------------------------
yaml_path = '/content/drive/MyDrive/Embedded/dataset/data.yaml'
project_path = '/content/drive/MyDrive/Embedded/training_results.1'
run_name = 'yolo_esp32_final'

# 2. EĞİTİMİ BAŞLAT
# ------------------------------------------
print("🚀 YOLOv8 Eğitimi Başlıyor (50 Epoch)...")

# Nano modeli yükle
model = YOLO('yolov8n.pt')

# Eğit
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=320,
    batch=16,
    patience=15,     # Early stopping
    name=run_name,   # Klasör ismi
    project=project_path, # Kayıt yeri
    device=0         # GPU
)

print("✅ Eğitim Başarıyla Tamamlandı!")

# 3. TFLITE FORMATINA ÇEVİR
# ------------------------------------------
print("🔄 En iyi model TFLite formatına çevriliyor...")

# Eğitimden çıkan en iyi ağırlıkları (best.pt) bul
best_weights = os.path.join(project_path, run_name, 'weights', 'best.pt')

# Dönüştürülecek modeli yükle
export_model = YOLO(best_weights)

# Çevir (320x320 boyutunda)
export_model.export(format='tflite', imgsz=320)

print(f"\n🎉 İŞLEM BİTTİ! Dosyanız hazır:")
print(f"📂 {best_weights.replace('.pt', '.tflite')}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 YOLOv8 Eğitimi Başlıyor (50 Epoch)...
Ultralytics 8.4.3 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Embedded/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 1.2s, saved as '/content/drive/MyDrive/Embedded/training_results/yolo_esp32_final/weights/best.onnx' (11.6 MB)
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /content/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 38.6files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/Embedded/training_results/yolo_esp32_final/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 320, 320, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 14, 2100), dtype=tf.float32, name=None)
Captures:
  136702670928400: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  136702670928592: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  136702649204816: TensorSpec(shape=(16,), dtype=tf.float32, na

In [3]:
from ultralytics import YOLO
import os

print("🚀 Model Eğitimi Başlıyor...")

# Yolu tanımla
yaml_path = '/content/drive/MyDrive/Embedded/dataset/data.yaml'

# Modeli Yükle (Nano)
model = YOLO('yolov8n.pt')

# Eğit
model.train(
    data=yaml_path,
    epochs=50,
    imgsz=320,
    batch=16,
    patience=15,
    name='yolo_esp32_final',
    project='/content/drive/MyDrive/Embedded/training_results.1',
    device=0
)

print("✅ Eğitim Bitti. TFLite Dönüştürülüyor...")

# En iyi modeli al ve dönüştür
best_pt = '/content/drive/MyDrive/Embedded/training_results/yolo_esp32_final/weights/best.pt'
model = YOLO(best_pt)
model.export(format='tflite', imgsz=320)

print(f"\n📂 TFLite Dosyan Hazır: {best_pt.replace('.pt', '.tflite')}")

🚀 Model Eğitimi Başlıyor...
Ultralytics 8.4.3 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Embedded/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_esp32_final, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, over

In [5]:
# TFLite dosyasını C Header dosyasına çevirme (xxd alternatifi)

tflite_path = '/content/drive/MyDrive/Embedded/training_results.1/yolo_esp32_final/weights/best.tflite'
output_c_path = '/content/drive/MyDrive/Embedded/training_results.1/model_data.h'

def tflite_to_c_array(tflite_path, output_path):
    with open(tflite_path, 'rb') as f:
        data = f.read()

    with open(output_path, 'w') as f:
        f.write('#ifndef MODEL_DATA_H\n')
        f.write('#define MODEL_DATA_H\n\n')
        f.write(f'unsigned char model_data[] = {{\n')

        for i, val in enumerate(data):
            f.write(f'0x{val:02x}, ')
            if (i + 1) % 12 == 0:
                f.write('\n')

        f.write('\n};\n\n')
        f.write(f'unsigned int model_data_len = {len(data)};\n')
        f.write('#endif\n')

print(f"🔄 Dönüştürülüyor: {tflite_path}")
try:
    tflite_to_c_array(tflite_path, output_c_path)
    print(f"✅ BAŞARILI! C Header dosyan şurada: {output_c_path}")
    print("ESP32 projen için bu dosyayı indirip kullanacaksın.")
except Exception as e:
    print(f"❌ Hata: {e}")

🔄 Dönüştürülüyor: /content/drive/MyDrive/Embedded/training_results.1/yolo_esp32_final/weights/best.tflite
❌ Hata: [Errno 2] No such file or directory: '/content/drive/MyDrive/Embedded/training_results.1/yolo_esp32_final/weights/best.tflite'


In [6]:
import os
from ultralytics import YOLO

# ==========================================
# 1. BEST.PT DOSYASINI ARA VE BUL
# ==========================================
search_dir = '/content/drive/MyDrive/Embedded/training_results.1'
print(f"🔍 '{search_dir}' içinde 'best.pt' aranıyor...")

best_pt_path = None
for root, dirs, files in os.walk(search_dir):
    if 'best.pt' in files:
        best_pt_path = os.path.join(root, 'best.pt')
        print(f"✅ BULUNDU: {best_pt_path}")
        break  # İlk bulduğunu al

if not best_pt_path:
    print("❌ HATA: 'best.pt' dosyası bulunamadı! Eğitim tamamlanmamış olabilir.")
else:
    # ==========================================
    # 2. TFLITE FORMATINA ÇEVİR (YENİDEN)
    # ==========================================
    print("\n🔄 TFLite dönüşümü başlatılıyor...")
    try:
        model = YOLO(best_pt_path)
        # int8 quantization ESP32 için kritiktir ama hata verirse parametreyi sil: int8=True
        model.export(format='tflite', imgsz=320)

        # Oluşan dosyanın yolu
        tflite_path = best_pt_path.replace('.pt', '.tflite')

        if os.path.exists(tflite_path):
            print(f"✅ TFLite oluşturuldu: {tflite_path}")

            # ==========================================
            # 3. C HEADER (.h) FORMATINA ÇEVİR
            # ==========================================
            print("\n📝 C Header (.h) dosyası hazırlanıyor...")
            output_c_path = os.path.join(os.path.dirname(tflite_path), 'model_data.h')

            with open(tflite_path, 'rb') as f:
                data = f.read()

            with open(output_c_path, 'w') as f:
                f.write('#ifndef MODEL_DATA_H\n')
                f.write('#define MODEL_DATA_H\n\n')
                f.write(f'unsigned char model_data[] = {{\n')

                for i, val in enumerate(data):
                    f.write(f'0x{val:02x}, ')
                    if (i + 1) % 12 == 0:
                        f.write('\n')

                f.write('\n};\n\n')
                f.write(f'unsigned int model_data_len = {len(data)};\n')
                f.write('#endif\n')

            print(f"🎉 İŞLEM BAŞARILI! Dosyan hazır:")
            print(f"📂 {output_c_path}")
            print("(Bu dosyayı indirip ESP32 projenin içine atacaksın)")

        else:
            print("❌ HATA: Export işlemi bitti ama .tflite dosyası oluşmadı.")

    except Exception as e:
        print(f"❌ BEKLENMEDİK HATA: {e}")

🔍 '/content/drive/MyDrive/Embedded/training_results.1' içinde 'best.pt' aranıyor...
✅ BULUNDU: /content/drive/MyDrive/Embedded/training_results.1/yolo_esp32_final/weights/best.pt

🔄 TFLite dönüşümü başlatılıyor...
Ultralytics 8.4.3 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Embedded/training_results.1/yolo_esp32_final/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 14, 2100) (5.9 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.20.1 opset 22...
ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 1.3s, saved as '/content/drive/MyDrive/Embedded/training_results.1/yolo_esp32_final/weights/best.onnx' (11.6 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/Embedded/training_results.

In [7]:
import os
import shutil

# ==========================================
# 1. DOSYALARI BULMA AYARLARI
# ==========================================
# Senin log çıktındaki yolu baz alıyoruz
base_weights_dir = '/content/drive/MyDrive/Embedded/training_results.1/yolo_esp32_final/weights'

# Olası TFLite yolları (YOLO bazen klasör içine, bazen dışına atar)
possible_paths = [
    os.path.join(base_weights_dir, 'best.tflite'),
    os.path.join(base_weights_dir, 'best_saved_model', 'best_float32.tflite'),
    os.path.join(base_weights_dir, 'best_saved_model', 'best_float16.tflite'),
]

tflite_path = None

print("🔍 TFLite dosyası aranıyor...")
for path in possible_paths:
    if os.path.exists(path):
        tflite_path = path
        print(f"✅ BULUNDU: {tflite_path}")
        break

if not tflite_path:
    print(f"❌ HATA: TFLite dosyası '{base_weights_dir}' ve alt klasörlerinde bulunamadı.")
else:
    # ==========================================
    # 2. C HEADER (.h) FORMATINA ÇEVİR
    # ==========================================
    print("\n📝 C Header (.h) dosyası hazırlanıyor...")

    # Header dosyasını 'weights' klasörüne kaydedelim
    output_c_path = os.path.join(base_weights_dir, 'model_data.h')

    try:
        with open(tflite_path, 'rb') as f:
            data = f.read()

        with open(output_c_path, 'w') as f:
            f.write('#ifndef MODEL_DATA_H\n')
            f.write('#define MODEL_DATA_H\n\n')
            f.write(f'// Model Boyutu: {len(data)} bytes\n')
            f.write(f'unsigned char model_data[] = {{\n')

            for i, val in enumerate(data):
                f.write(f'0x{val:02x}, ')
                if (i + 1) % 12 == 0:
                    f.write('\n')

            f.write('\n};\n\n')
            f.write(f'unsigned int model_data_len = {len(data)};\n')
            f.write('#endif\n')

        print(f"🎉 İŞLEM BAŞARILI! Dosyan hazır:")
        print(f"📂 {output_c_path}")
        print("(Bu model_data.h dosyasını indirip ESP32 projenin içine atacaksın)")

    except Exception as e:
        print(f"❌ Dönüştürme Hatası: {e}")

🔍 TFLite dosyası aranıyor...
✅ BULUNDU: /content/drive/MyDrive/Embedded/training_results.1/yolo_esp32_final/weights/best_saved_model/best_float32.tflite

📝 C Header (.h) dosyası hazırlanıyor...
🎉 İŞLEM BAŞARILI! Dosyan hazır:
📂 /content/drive/MyDrive/Embedded/training_results.1/yolo_esp32_final/weights/model_data.h
(Bu model_data.h dosyasını indirip ESP32 projenin içine atacaksın)
